### Математический аппарат алгоритма AdaBoost (Бинарная классификация)

Целевые метки $y$ и предсказания моделей $h(x)$ принимают значения $1$ или $-1$.

#### 1. Инициализация весов объектов
Перед стартом обучения всем $N$ объектам выборки присваиваются одинаковые стартовые веса:
$$w_i^{(1)} = \frac{1}{N}, \quad i = 1, \dots, N$$

#### 2. Расчет взвешенной ошибки базовой модели ($\epsilon_t$)
На итерации $t$ базовая модель обучается с учетом весов объектов. Общая ошибка модели считается как сумма весов тех объектов, на которых модель споткнулась:
$$\epsilon_t = \sum_{i: y_i \neq h_t(x_i)} w_i^{(t)}$$

#### 3. Расчет веса (авторитета) модели в ансамбле ($\alpha_t$)
Коэффициент важности («сила голоса») модели рассчитывается на основе её ошибки. Дополнительно масштабируется параметром скорости обучения ($\eta$ — `learning_rate`):
$$\alpha_t = \eta \cdot \frac{1}{2} \ln\left(\frac{1 - \epsilon_t}{\epsilon_t + 10^{-10}}\right)$$
*Константа $10^{-10}$ защищает от деления на ноль, если модель безошибочна ($\epsilon_t = 0$).*

#### 4. Обновление весов объектов данных
Для следующей итерации веса объектов пересчитываются. Ошибочным объектам вес увеличивается, угаданным — уменьшается:
$$w_i^{(t+1)} = w_i^{(t)} \cdot \exp\left(-\alpha_t \cdot y_i \cdot h_t(x_i)\right)$$

*   Если модель **угадала** ($y_i = h_t(x_i)$): произведение $y_i \cdot h_t(x_i) = 1$, вес умножается на $e^{-\alpha_t}$ (уменьшается).
*   Если модель **ошиблась** ($y_i \neq h_t(x_i)$): произведение $y_i \cdot h_t(x_i) = -1$, минус на минус даёт плюс, вес умножается на $e^{\alpha_t}$ (увеличивается).

#### 5. Нормализация весов
Чтобы веса оставались распределением вероятностей (их сумма всегда равнялась $1$), на каждом шаге выполняется нормировка:
$$w_i^{(t+1)} = \frac{w_i^{(t+1)}}{\sum_{j=1}^{N} w_j^{(t+1)}}$$

#### 6. Финальное взвешенное голосование ансамбля (Predict)
Итоговое предсказание вычисляется как знак от взвешенной суммы ответов всех построенных базовых моделей ($T$ — общее число моделей):
$$H(x) = \text{sign}\left( \sum_{t=1}^{T} \alpha_t \cdot h_t(x) \right)$$


In [53]:
import numpy as np
from sklearn.tree import DecisionTreeClassifier
from sklearn.base import clone

class AdaBoost:
    def __init__(self,model = DecisionTreeClassifier(max_depth=1), n_estimators = 100, learning_rate = 1):
        """
        Реализует алгоритм адаптивного бустинга AdaBoost (алгоритм SAMME) для задач бинарной классификации.

        Модель последовательно строит ансамбль из слабых учеников (по умолчанию — пни решений глубиной 1), 
        каждый из которых обучается на данных с измененными весами объектов. Объекты, на которых предыдущее 
        дерево ошиблось, получают больший вес на следующей итерации. Итоговое предсказание формируется 
        путем взвешенного голосования всех моделей на основе их индивидуального авторитета (альфа).
        
        Аргументы инициализации:
        ------------------------
        model : estimator object, default=DecisionTreeClassifier(max_depth=1)
            Базовая модель («слабый ученик»), на основе которой строится ансамбль.
            Должна поддерживать параметр sample_weight в методе fit.
        n_estimators : int, default=100
            Максимальное количество последовательных моделей (эпох бустинга) в ансамбле.
        learning_rate : float, default=1.0
            Скорость обучения (коэффициент усадки), масштабирующая вклад каждой новой модели.
        
        Основные методы:
        ----------------
        fit(X, y)
            Запускает итерационный цикл построения ансамбля. Вычисляет взвешенную ошибку
            каждого дерева, рассчитывает его авторитет (альфа) и обновляет веса объектов
            экспоненциальным методом. Поддерживает критерий ранней остановки.
        predict(X)
            Предсказывает финальные метки классов (0 или 1) на основе взвешенной суммы
            голосов всех сохраненных базовых моделей ансамбля.
        
        Внутренние атрибуты:
        --------------------
        self.base_model : list
            Список объектов успешно обученных базовых моделей (деревьев решений) на каждой эпохе.
        self.weight_model : list
            Список рассчитанных значений авторитета (альфа) для каждого базового дерева.
        self.w : numpy.ndarray
            Текущий вектор весов объектов обучающей выборки. Длина равна количеству строк в X.
        self.history_w_models : list
            Список снимков вектора весов объектов self.w на каждой итерации для мониторинга.
        self.history_epsilon : list
            Cписок взвешенных ошибок (epsilon) моделей на каждой итерации.
"""
        self.model = model
        self.n_estimators = n_estimators
        self.learning_rate = learning_rate
        self.base_model = []    # Для хранения моделей
        self.weight_model = []  # Для весов модели
        self.history_w_models = []
        self.history_epsilon = []

    def fit(self,X,y):
        X = np.asarray(X)
        y = np.where(y == 1, 1, -1)
        
        self.w = np.ones(X.shape[0])
        self.w /= X.shape[0]

        for epoch in range(self.n_estimators):
            tree = clone(self.model)
            tree.fit(X,y, sample_weight=self.w)
            pred = tree.predict(X)
            pred = np.where(pred==1,1,-1)

            epsilon = np.sum(self.w[y != pred])
            alpha = self.learning_rate * (1/2) * np.log((1.0 - epsilon) / (epsilon + 1e-10))  # 1e-10 исп для избежания деления на 0

            self.history_epsilon.append(epsilon)

            if epsilon >= 0.5 or epsilon == 0:  # 0 -дерево обучилось абсолютно идеально, >0.5 модель угадывает ответы ровно в половине случаев
                break
                
            self.base_model.append(tree)
            self.weight_model.append(alpha)

            correct = (y == pred)
            mis = (y != pred)
            
            self.w[correct] *= np.exp(-alpha)  # Для верных значений
            self.w[mis] *= np.exp(alpha)  # Для ошибочных умножаем на e^(alpha)

            self.w /= np.sum(self.w)

            self.history_w_models.append(self.w.copy())
        self.epoch = len(self.base_model)
        return self
            
    def predict(self,X):
        X = np.asarray(X)

        model_vote = np.zeros(X.shape[0])

        for model,weight in zip(self.base_model, self.weight_model):
            pred = model.predict(X)
            pred = np.where(pred==1,1,-1) 
            model_vote += weight * pred
            
        return np.where(model_vote >= 0, 1, 0)


In [54]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.ensemble import AdaBoostClassifier 

data = load_breast_cancer()
X, y = data.data, data.target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

custom_model = AdaBoost(n_estimators=50, learning_rate=0.5)
custom_model.fit(X_train, y_train)
custom_preds = custom_model.predict(X_test)

sklearn_model = AdaBoostClassifier(n_estimators=50, learning_rate=0.5, random_state=42)
sklearn_model.fit(X_train, y_train)
sklearn_preds = sklearn_model.predict(X_test)


print(f"Кастомный AdaBoost Accuracy: {accuracy_score(y_test, custom_preds):.4f}")
print(f"Sklearn AdaBoost Classifier Accuracy:  {accuracy_score(y_test, sklearn_preds):.4f}")
print(f"Количество успешно обученных эпох:      {(custom_model.epoch)}")


Кастомный AdaBoost Accuracy: 0.9649
Sklearn AdaBoost Classifier Accuracy:  0.9649
Количество успешно обученных эпох:      50


### Cравнение веса моделей при принятии решения

In [55]:
print('Кастомный AdaBoost')
for err in (custom_model.weight_model[:10]):
    print(round(err,3), end = ', ')
print()
print()
print('Sklearn Adaboost')
for err in (sklearn_model.estimator_weights_ / 2)[:10]:
    print(round(err,3), end = ', ')

Кастомный AdaBoost
0.614, 0.541, 0.367, 0.322, 0.29, 0.253, 0.23, 0.266, 0.238, 0.198, 

Sklearn Adaboost
0.614, 0.541, 0.367, 0.322, 0.29, 0.253, 0.23, 0.266, 0.238, 0.198, 

### Сравнение взвешенных ошибок моделей на каждом шаге (epsilon) sklearn и кастомной модели 

In [56]:
print(f'    Ошибки sklearn модели')
for eps in sklearn_model.estimator_errors_[:10]:
    print(round(eps,3), end = ', ')
print()
print()
print(f'    Ошибки кастомной модели')
for eps in custom_model.history_epsilon[:10]:
    print(round(eps,3), end = ', ')

    Ошибки sklearn модели
0.079, 0.103, 0.187, 0.216, 0.239, 0.267, 0.285, 0.256, 0.279, 0.312, 

    Ошибки кастомной модели
0.079, 0.103, 0.187, 0.216, 0.239, 0.267, 0.285, 0.256, 0.279, 0.312, 

In [57]:
for cnt_model in range(1,11):
    model = AdaBoost(n_estimators=cnt_model, learning_rate=0.5)
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    print(f'''На {cnt_model} модели точность составляет {accuracy_score(y_test, pred):.4f}''')

На 1 модели точность составляет 0.8947
На 2 модели точность составляет 0.8947
На 3 модели точность составляет 0.9386
На 4 модели точность составляет 0.9211
На 5 модели точность составляет 0.9561
На 6 модели точность составляет 0.9386
На 7 модели точность составляет 0.9561
На 8 модели точность составляет 0.9386
На 9 модели точность составляет 0.9561
На 10 модели точность составляет 0.9561


##  История изменения весов первых пяти объектов выборки на протяжении первых пяти эпох обучения

In [58]:
for i in range(5):
    print(model.history_w_models[i][:5])

[0.00184564 0.00184564 0.00184564 0.00184564 0.00184564]
[0.00153675 0.00153675 0.00153675 0.00153675 0.00153675]
[0.00265993 0.00127763 0.00127763 0.00127763 0.00127763]
[0.00222487 0.00106865 0.00106865 0.00106865 0.00106865]
[0.00187343 0.00089985 0.00089985 0.00160701 0.00089985]


Поскольку изначально сумма 1.0 делилась на всю большую выборку из 455 объектов ($1 / 455 \approx 0.00219$) - вес каждого обьекта данных

### Анализ изменения весов объектов по эпохам

1. **Строка 1 (Эпоха 1):** `[0.00184, 0.00184, 0.00184, 0.00184, 0.00184]`
   * **Что произошло:** Первое базовое дерево обучилось и правильно предсказало все пять объектов (их веса одинаково снизились с исходных $0.00219$ до $0.00184$).
   * **Логика:** Поскольку эти объекты простые, алгоритм уменьшил внимание к ним, перекинув освободившуюся долю общего веса на другие объекты датасета, в которых дерево запуталось.

2. **Строка 2 (Эпоха 2):** `[0.00153, 0.00153, 0.00153, 0.00153, 0.00153]`
   * **Что произошло:** Второе дерево тоже оказалось успешным для этой пятерки. Веса всех пяти объектов снова синхронно уменьшились (с $0.00184$ до $0.00153$).
   * **Логика:** Ансамбль продолжает легко справляться с этой группой примеров, их «важность» падает.

3. **Строка 3 (Эпоха 3):** `[0.00265, 0.00127, 0.00127, 0.00127, 0.00127]`
   * **Что произошло:** Произошел переломный момент. Третье дерево ошиблось на самом первом объекте, но угадало остальные четыре.
   * **Логика:** Вес первого объекта резко подскочил почти в два раза ($0.00153 \rightarrow 0.00265$). Теперь он стал для алгоритма приоритетным, а веса остальных четырех объектов продолжили падать ($0.00127$).

4. **Строка 4 (Эпоха 4):** `[0.00222, 0.00106, 0.00106, 0.00106, 0.00106]`
   * **Что произошло:** Четвертое дерево учло тяжелый вес первого объекта и успешно исправило ошибку предшественника, правильно предсказав всю пятерку.
   * **Логика:** Вес первого объекта снова пошел на спад ($0.00265 \rightarrow 0.00222$), как и веса остальных ($0.00106$).

5. **Строка 5 (Эпоха 5):** `[0.00187, 0.00089, 0.00089, 0.00160, 0.00089]`
   * **Что произошло:** Пятое дерево без проблем справилось с объектами 1, 2, 3 и 5 (их веса снизились), но допустило ошибку на четвертом объекте.
   * **Логика:** Вес четвертого объекта мгновенно вырос с $0.00106$ до $0.00160$. На следующей (шестой) итерации новое дерево будет вынуждено подстраиваться под него.


In [59]:
print(f"Веса первых 5 моделей (alpha): ")
for weight_model in model.weight_model[:5]:
    print(round(weight_model,2))


Веса первых 5 моделей (alpha): 
0.61
0.54
0.37
0.32
0.29


Самым авторитетным оказалось самое первое дерево так как пенек ищет самый очевидный, самый сильный и глобальный признак в датасете, который разделяет раковые опухоли лучше всего. Он делает максимально «полезный» срез данных с минимальной ошибкой ε, поэтому его финальный авторитет α получается самым высоким.

По мере движения происходит классический процесс бустинга - Первые деревья забирают себе самые простые закономерности -> Ошибка ε новых деревьев неизбежно растет, так как им приходится решать заведомо более сложную «запутанную» задачу -> чем выше ошибка ε, тем меньше становится авторитет модели α по формуле логарифма

При вызове predict мнение первого дерева имеет огромный вес, но если последующие 2–3 дерева (например, второе с 0.54 и третье с 0.36, которые в сумме дают 0.90) дружно проголосуют против первого, они легко перевесят его голос. В этом и заключается сила ансамбля: авторитетное большинство исправляет ошибки сильного, но локального лидера.